# Spark Streaming: Kafka → ClickHouse/Citus

This notebook demonstrates:
1. Spark native Kubernetes mode
2. Streaming consumption from Kafka topics
3. Writing to ClickHouse (metrics) and Citus (workouts)
4. Monitoring executor pod creation

**Topics:**
- `metrics`: athlete_id, timestamp, power, heart_rate, cadence, speed, distance
- `workouts`: athlete_id, workout_id, date, duration, status

In [ ]:
# 1. Initialize Spark Session in Native Kubernetes Mode
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, window, avg, max as spark_max, min as spark_min
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType, FloatType
import os

# Check environment
print("Environment Variables:")
print(f"CITUS_HOST: {os.getenv('CITUS_HOST', 'citus-coordinator.bigintensive.svc.cluster.local')}")
print(f"KAFKA_BOOTSTRAP_SERVERS: {os.getenv('KAFKA_BOOTSTRAP_SERVERS', 'kafka:19092')}")
print(f"CLICKHOUSE_HOST: {os.getenv('CLICKHOUSE_HOST', 'clickhouse-client.bigintensive.svc.cluster.local')}")

In [ ]:
# 2. Create SparkSession with Native Kubernetes Configuration
spark = SparkSession.builder \
    .appName("BigIntensive-Kafka-Streaming") \
    .master("k8s://https://kubernetes.default:443") \
    .config("spark.kubernetes.namespace", "bigintensive") \
    .config("spark.kubernetes.authenticate.driver.mounted", "true") \
    .config("spark.kubernetes.driver.pod.name", os.getenv('HOSTNAME', 'jupyter-0')) \
    .config("spark.kubernetes.executor.request.cores", "0.5") \
    .config("spark.kubernetes.executor.limit.cores", "1") \
    .config("spark.kubernetes.executor.request.memory", "1G") \
    .config("spark.kubernetes.executor.limit.memory", "2G") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "1") \
    .config("spark.dynamicAllocation.maxExecutors", "4") \
    .config("spark.dynamicAllocation.executorIdleTimeout", "60s") \
    .config("spark.dynamicAllocation.cachedExecutorIdleTimeout", "120s") \
    .getOrCreate()

spark.sparkContext.setLogLevel("INFO")
print(f"\nSpark Version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"App Name: {spark.sparkContext.appName}")

## Consumer 1: Kafka "metrics" → ClickHouse

In [ ]:
# 3. Define Metrics Schema
metrics_schema = StructType([
    StructField("athlete_id", IntegerType()),
    StructField("workout_id", IntegerType()),
    StructField("timestamp", TimestampType()),
    StructField("power", FloatType()),
    StructField("heart_rate", IntegerType()),
    StructField("cadence", IntegerType()),
    StructField("speed", FloatType()),
    StructField("distance", FloatType())
])

# Read from Kafka topic "metrics"
kafka_bootstrap = os.getenv('KAFKA_BOOTSTRAP_SERVERS', 'kafka:19092')

df_metrics_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap) \
    .option("subscribe", "metrics") \
    .option("startingOffsets", "latest") \
    .load()

# Parse JSON
df_metrics = df_metrics_raw.select(
    from_json(col("value").cast("string"), metrics_schema).alias("data")
).select("data.*")

print("\nMetrics Schema:")
df_metrics.printSchema()

In [ ]:
# 4. (Optional) Aggregate Metrics with 5-minute Window
df_metrics_agg = df_metrics.groupBy(
    col("athlete_id"),
    window(col("timestamp"), "5 minutes", "5 minutes")
).agg(
    avg("power").alias("avg_power"),
    spark_max("power").alias("max_power"),
    spark_min("power").alias("min_power"),
    avg("heart_rate").alias("avg_heart_rate"),
    avg("speed").alias("avg_speed"),
    spark_max("distance").alias("total_distance")
)

print("\nAggregated Metrics Schema:")
df_metrics_agg.printSchema()

In [ ]:
# 5. Write Metrics to ClickHouse
# Note: Using HTTP API via clickhouse-client driver
# For production, consider using ClickHouse native protocol

def write_metrics_to_clickhouse(df, batch_id):
    """Write metrics batch to ClickHouse"""
    if df.count() == 0:
        print(f"Batch {batch_id}: No metrics to write")
        return
    
    clickhouse_host = os.getenv('CLICKHOUSE_HOST', 'clickhouse-client.bigintensive.svc.cluster.local')
    clickhouse_port = 8123
    
    # For now, just show sample data
    print(f"\n=== Batch {batch_id}: Writing {df.count()} metrics ===")
    df.show(5, truncate=False)
    
    # In production, write via ClickHouse driver or HTTP API
    # df.write \
    #     .format("clickhouse") \
    #     .option("url", f"http://{clickhouse_host}:{clickhouse_port}") \
    #     .option("database", "bigintensive") \
    #     .option("table", "workout_metrics") \
    #     .mode("append") \
    #     .save()

# Start streaming to ClickHouse
query_metrics = df_metrics.writeStream \
    .foreachBatch(write_metrics_to_clickhouse) \
    .option("checkpointLocation", "/tmp/spark_checkpoint_metrics") \
    .start()

print("✅ Metrics streaming started!")
print(f"Query ID: {query_metrics.id}")

## Consumer 2: Kafka "workouts" → Citus

In [ ]:
# 6. Define Workouts Schema
workouts_schema = StructType([
    StructField("athlete_id", IntegerType()),
    StructField("workout_id", IntegerType()),
    StructField("date", StringType()),
    StructField("duration_minutes", IntegerType()),
    StructField("status", StringType())  # completed, cancelled
])

# Read from Kafka topic "workouts"
df_workouts_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap) \
    .option("subscribe", "workouts") \
    .option("startingOffsets", "latest") \
    .load()

# Parse JSON
df_workouts = df_workouts_raw.select(
    from_json(col("value").cast("string"), workouts_schema).alias("data")
).select("data.*")

print("\nWorkouts Schema:")
df_workouts.printSchema()

In [ ]:
# 7. Write Workouts to Citus
def write_workouts_to_citus(df, batch_id):
    """Write workouts batch to Citus"""
    if df.count() == 0:
        print(f"Batch {batch_id}: No workouts to write")
        return
    
    citus_host = os.getenv('CITUS_HOST', 'citus-coordinator.bigintensive.svc.cluster.local')
    citus_port = 5432
    citus_user = os.getenv('CITUS_USER', 'postgres')
    citus_password = os.getenv('CITUS_PASSWORD', 'postgres')
    
    print(f"\n=== Batch {batch_id}: Writing {df.count()} workouts ===")
    df.show(5, truncate=False)
    
    # Write via JDBC to Citus
    try:
        df.write \
            .format("jdbc") \
            .option("url", f"jdbc:postgresql://{citus_host}:{citus_port}/bigintensive") \
            .option("dbtable", "workouts") \
            .option("user", citus_user) \
            .option("password", citus_password) \
            .option("driver", "org.postgresql.Driver") \
            .mode("append") \
            .save()
        print(f"✅ Batch {batch_id} written to Citus successfully")
    except Exception as e:
        print(f"⚠️ Error writing to Citus: {e}")

# Start streaming to Citus
query_workouts = df_workouts.writeStream \
    .foreachBatch(write_workouts_to_citus) \
    .option("checkpointLocation", "/tmp/spark_checkpoint_workouts") \
    .start()

print("✅ Workouts streaming started!")
print(f"Query ID: {query_workouts.id}")

## Monitoring

In [ ]:
# 8. Monitor Active Queries
import time

print("\n=== Active Streaming Queries ===")
for query in spark.streams.active:
    print(f"\nQuery: {query.name}")
    print(f"ID: {query.id}")
    print(f"Status: {query.status}")
    print(f"Latest Progress: {query.lastProgress}")

In [ ]:
# 9. Check Executor Pods
import subprocess
import time

print("\n=== Checking Spark Executor Pods ===")
print("Run this in a terminal to watch executor creation:")
print("  kubectl get pods -n bigintensive -l spark-role=executor -w")
print("\nExecutor pods will appear/disappear as Spark processes data")

try:
    result = subprocess.run(
        ["kubectl", "get", "pods", "-n", "bigintensive", "-l", "spark-role=executor", "-o", "wide"],
        capture_output=True,
        text=True,
        timeout=5
    )
    print("\nCurrent Executor Pods:")
    print(result.stdout)
except Exception as e:
    print(f"Could not fetch pods: {e}")

## Testing with Sample Data

In [ ]:
# 10. Generate Sample Metrics Data to Kafka
from kafka import KafkaProducer
import json
from datetime import datetime, timedelta
import random

try:
    producer = KafkaProducer(
        bootstrap_servers=[kafka_bootstrap],
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )
    
    # Generate 10 sample metrics
    print("\n=== Generating Sample Metrics ===")
    for i in range(10):
        athlete_id = random.randint(1, 100)
        workout_id = random.randint(1, 1000)
        timestamp = (datetime.now() - timedelta(minutes=random.randint(0, 5))).isoformat()
        
        metric = {
            "athlete_id": athlete_id,
            "workout_id": workout_id,
            "timestamp": timestamp,
            "power": random.uniform(150, 400),
            "heart_rate": random.randint(120, 180),
            "cadence": random.randint(80, 100),
            "speed": random.uniform(25, 45),
            "distance": random.uniform(1, 5)
        }
        
        producer.send("metrics", value=metric)
        print(f"Sent metric {i+1}: Athlete {athlete_id}, Power {metric['power']:.1f}W")
    
    producer.flush()
    producer.close()
    print("\n✅ Sample metrics sent to Kafka!")
    
except Exception as e:
    print(f"⚠️ Error sending to Kafka: {e}")

In [ ]:
# 11. Generate Sample Workouts Data to Kafka
try:
    producer = KafkaProducer(
        bootstrap_servers=[kafka_bootstrap],
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )
    
    # Generate 5 sample workouts
    print("\n=== Generating Sample Workouts ===")
    for i in range(5):
        athlete_id = random.randint(1, 100)
        workout_id = random.randint(1, 1000)
        date = (datetime.now() - timedelta(days=random.randint(0, 7))).strftime('%Y-%m-%d')
        duration = random.randint(30, 180)
        status = random.choice(["completed", "cancelled"])
        
        workout = {
            "athlete_id": athlete_id,
            "workout_id": workout_id,
            "date": date,
            "duration_minutes": duration,
            "status": status
        }
        
        producer.send("workouts", value=workout)
        print(f"Sent workout {i+1}: Athlete {athlete_id}, {duration}min, Status {status}")
    
    producer.flush()
    producer.close()
    print("\n✅ Sample workouts sent to Kafka!")
    
except Exception as e:
    print(f"⚠️ Error sending to Kafka: {e}")

## Verify Data in Databases

In [ ]:
# 12. Verify Data in Citus
from psycopg2 import connect

try:
    citus_host = os.getenv('CITUS_HOST', 'citus-coordinator.bigintensive.svc.cluster.local')
    conn = connect(
        host=citus_host,
        port=5432,
        database="bigintensive",
        user="postgres",
        password="postgres"
    )
    cursor = conn.cursor()
    
    cursor.execute("SELECT COUNT(*) FROM workouts")
    count = cursor.fetchone()[0]
    print(f"✅ Workouts in Citus: {count}")
    
    cursor.execute("SELECT * FROM workouts LIMIT 3")
    rows = cursor.fetchall()
    print("\nSample workouts:")
    for row in rows:
        print(f"  Athlete {row[0]}, Workout {row[1]}, Status {row[4]}")
    
    cursor.close()
    conn.close()
    
except Exception as e:
    print(f"⚠️ Could not connect to Citus: {e}")

In [ ]:
# 13. Verify Data in ClickHouse
try:
    clickhouse_host = os.getenv('CLICKHOUSE_HOST', 'clickhouse-client.bigintensive.svc.cluster.local')
    
    # Using clickhouse-driver if available
    from clickhouse_driver import Client
    
    client = Client(clickhouse_host)
    count = client.execute("SELECT COUNT(*) FROM bigintensive.workout_metrics")[0][0]
    print(f"✅ Metrics in ClickHouse: {count}")
    
    rows = client.execute("SELECT athlete_id, power, heart_rate FROM bigintensive.workout_metrics LIMIT 3")
    print("\nSample metrics:")
    for row in rows:
        print(f"  Athlete {row[0]}, Power {row[1]:.1f}W, HR {row[2]}bpm")
    
except ImportError:
    print("⚠️ clickhouse-driver not installed. Install with: pip install clickhouse-driver")
except Exception as e:
    print(f"⚠️ Could not connect to ClickHouse: {e}")

## Stop Streaming (when done)

In [ ]:
# 14. Stop Streaming Queries
print("=== Stopping Streaming Queries ===")

for query in spark.streams.active:
    print(f"Stopping {query.name}...")
    query.stop()
    query.awaitTermination()
    print(f"✅ {query.name} stopped")

print("\n✅ All queries stopped")